In [0]:
# ============================================================
# GOLD LAYER - SUMMARY TABLES
# ============================================================

from pyspark.sql.functions import (
    sum,
    avg,
    countDistinct,
    col,
    when
)

# GOLD LAYER - READ SILVER DATA
silver = spark.table("silver_retail")

In [0]:
# MONTHLY SALES
gold_monthly_sales = (
    silver
    .groupBy("Year", "Month")
    .agg(
        sum(
            when(col("Total") > 0, col("Total"))
            .otherwise(0)
        ).alias("Gross_Revenue"),

        sum(
            when(col("Total") < 0, -col("Total"))
            .otherwise(0)
        ).alias("Returns"),

        sum("Total").alias("Net_Revenue"),

        countDistinct(
            when(
                col("Quantity") > 0,
                col("InvoiceNo")
            )
        ).alias("Orders"),

        sum(
            when(col("Quantity") > 0, col("Quantity"))
            .otherwise(0)
        ).alias("Units_Sold")
    )
    .orderBy("Year", "Month")
)

gold_monthly_sales.show()

gold_monthly_sales.write.format("delta").mode("overwrite").saveAsTable("gold_monthly_sales")

+----+-----+------------------+------------------+------------------+------+----------+
|Year|Month|     Gross_Revenue|           Returns|       Net_Revenue|Orders|Units_Sold|
+----+-----+------------------+------------------+------------------+------+----------+
|2010|   12| 821452.7299999996| 74729.11999999976| 746723.6099999994|  1629|    361094|
|2011|    1|  689811.609999999| 131363.0499999999|  558448.559999999|  1120|    397030|
|2011|    2| 522545.5600000005|25519.150000000034| 497026.4100000005|  1126|    286074|
|2011|    3| 716215.2600000004| 34201.28000000008| 682013.9800000009|  1531|    384023|
|2011|    4| 536968.4910000009| 44600.65000000004|492367.84100000054|  1318|    311314|
|2011|    5| 769296.6100000005| 47202.50999999987| 722094.1000000008|  1731|    398686|
|2011|    6| 760547.0099999992| 70569.78000000003| 689977.2299999993|  1576|    393633|
|2011|    7|        718076.121|37919.130000000056| 680156.9910000004|  1540|    405473|
|2011|    8| 757841.3799999999|5

In [0]:
# DAILY SALES
gold_daily_sales = (
    silver
    .groupBy("Date")
    .agg(
        sum(
            when(col("Total") > 0, col("Total"))
            .otherwise(0)
        ).alias("Gross_Revenue"),

        sum(
            when(col("Total") < 0, -col("Total"))
            .otherwise(0)
        ).alias("Returns"),

        sum("Total").alias("Net_Revenue"),

        countDistinct(
            when(
                col("Quantity") > 0,
                col("InvoiceNo")
            )
        ).alias("Orders"),

        sum(
            when(col("Quantity") > 0, col("Quantity"))
            .otherwise(0)
        ).alias("Units_Sold")
    )
    .orderBy("Date")
)

gold_daily_sales.show(10)

gold_daily_sales.write.format("delta").mode("overwrite").saveAsTable("gold_daily_sales")

+----------+------------------+------------------+------------------+------+----------+
|      Date|     Gross_Revenue|           Returns|       Net_Revenue|Orders|Units_Sold|
+----------+------------------+------------------+------------------+------+----------+
|2010-12-01|58776.790000000015| 325.2299999999999| 58451.56000000002|   136|     26906|
|2010-12-02| 47629.41999999999|1541.1000000000008|46088.319999999985|   143|     31283|
|2010-12-03| 46898.63000000001|1323.2499999999998| 45575.38000000001|    73|     16430|
|2010-12-05|31364.630000000012|390.99999999999994|30973.630000000012|    88|     16243|
|2010-12-06| 54624.15000000003|            970.28| 53653.87000000003|   108|     21775|
|2010-12-07| 99553.85000000003| 54559.14999999998|44994.700000000055|    85|     25324|
|2010-12-08|45235.360000000015|1200.1399999999999|44035.220000000016|   123|     23049|
|2010-12-09|53548.189999999995|1054.0500000000002| 52494.13999999999|   132|     20698|
|2010-12-10| 59021.02000000001|1

In [0]:
# SALES BY COUNTRY
gold_sales_by_country = (
    silver
    .groupBy("Country")
    .agg(
        sum(
            when(col("Total") > 0, col("Total"))
            .otherwise(0)
        ).alias("Gross_Revenue"),

        sum(
            when(col("Total") < 0, -col("Total"))
            .otherwise(0)
        ).alias("Returns"),

        sum("Total").alias("Net_Revenue"),

        countDistinct(
            when(
                col("Quantity") > 0,
                col("InvoiceNo")
            )
        ).alias("Orders"),

        sum(
            when(col("Quantity") > 0, col("Quantity"))
            .otherwise(0)
        ).alias("Units_Sold")
    )
    .orderBy(col("Net_Revenue").desc())
)

gold_sales_by_country.show()

gold_sales_by_country.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_country")



+---------------+------------------+------------------+------------------+------+----------+
|        Country|     Gross_Revenue|           Returns|       Net_Revenue|Orders|Units_Sold|
+---------------+------------------+------------------+------------------+------+----------+
| United Kingdom| 9001744.093999987| 812491.7900000003|8189252.3040000405| 18784|   4718327|
|    Netherlands|         285446.34| 784.8000000000001|284661.54000000004|    95|    200937|
|           EIRE| 283140.5199999999| 20147.14000000001|262993.37999999995|   288|    147281|
|        Germany|228678.39999999976| 7168.929999999995| 221509.4699999998|   457|    119156|
|         France|209625.37000000005|12308.259999999993|197317.11000000007|   392|    112061|
|      Australia|138453.81000000003| 1444.040000000002|137009.77000000002|    57|     84199|
|    Switzerland|57067.600000000006|            704.55|          56363.05|    54|     30618|
|          Spain| 61558.55999999999| 6802.529999999999|54756.030000000

In [0]:
# SALES BY PRODUCT
gold_sales_by_product = (
    silver
    .groupBy("StockCode", "Description")
    .agg(
        sum(
            when(col("Total") > 0, col("Total"))
            .otherwise(0)
        ).alias("Gross_Revenue"),

        sum(
            when(col("Total") < 0, -col("Total"))
            .otherwise(0)
        ).alias("Returns"),

        sum("Total").alias("Net_Revenue"),

        sum(
            when(col("Quantity") > 0, col("Quantity"))
            .otherwise(0)
        ).alias("Units_Sold"),

        countDistinct(
            when(
                col("Quantity") > 0,
                col("InvoiceNo")
            )
        ).alias("Orders")
    )
    .orderBy(col("Net_Revenue").desc())
)

gold_sales_by_product.show(10)

gold_sales_by_product.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_product")


+---------+--------------------+------------------+------------------+------------------+----------+------+
|StockCode|         Description|     Gross_Revenue|           Returns|       Net_Revenue|Units_Sold|Orders|
+---------+--------------------+------------------+------------------+------------------+----------+------+
|      DOT|      DOTCOM POSTAGE|206248.77000000016|              3.29|206245.48000000016|       708|   708|
|    22423|REGENCY CAKESTAND...|174156.54000000036|           9697.05|164459.49000000028|     13862|  1989|
|    47566|       PARTY BUNTING| 99445.23000000052|           1201.35| 98243.88000000053|     18287|  1686|
|   85123A|WHITE HANGING HEA...| 104284.2399999991| 6624.299999999999| 97659.93999999916|     37584|  2193|
|   85099B|JUMBO BAG RED RET...| 94159.81000000134|1984.0199999999998| 92175.79000000127|     48375|  2092|
|    23084|  RABBIT NIGHT LIGHT| 66870.03000000006|208.40000000000003| 66661.63000000005|     30739|   994|
|     POST|             POST

In [0]:
# SALES BY CUSTOMER
gold_sales_by_customer = (
    silver
    .filter(col("CustomerID").isNotNull())
    .groupBy("CustomerID")
    .agg(
        sum(
            when(col("Total") > 0, col("Total"))
            .otherwise(0)
        ).alias("Gross_Revenue"),

        sum(
            when(col("Total") < 0, -col("Total"))
            .otherwise(0)
        ).alias("Returns"),

        sum("Total").alias("Net_Revenue"),

        sum(
            when(col("Quantity") > 0, col("Quantity"))
            .otherwise(0)
        ).alias("Units_Sold"),

        countDistinct(
            when(
                col("Quantity") > 0,
                col("InvoiceNo")
            )
        ).alias("Orders")
    )
    .orderBy(col("Net_Revenue").desc())
)

gold_sales_by_customer.show(10)

gold_sales_by_customer.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_customer")

+----------+------------------+------------------+------------------+----------+------+
|CustomerID|     Gross_Revenue|           Returns|       Net_Revenue|Units_Sold|Orders|
+----------+------------------+------------------+------------------+----------+------+
|   14646.0| 280206.0200000001| 717.0000000000001| 279489.0200000001|    197491|    74|
|   18102.0|259657.30000000002|           3218.81|256438.49000000002|     64124|    60|
|   17450.0|194390.78999999998| 7068.620000000001|187322.16999999998|     69973|    46|
|   14911.0|143711.16999999998|11252.439999999995|         132458.73|     80490|   201|
|   12415.0|124914.53000000006|1189.0800000000015|123725.45000000004|     77670|    21|
|   14156.0|117210.08000000003|3995.4900000000007|113214.59000000003|     57768|    55|
|   17511.0| 91062.37999999999|2937.0000000000005| 88125.37999999999|     64549|    31|
|   16684.0|          66653.56| 761.4799999999999| 65892.07999999999|     50255|    28|
|   13694.0| 65039.61999999999| 

In [0]:
# OVERALL KPIS
gold_overall_kpis = (
    silver
    .agg(
        sum(
            when(col("Total") > 0, col("Total"))
            .otherwise(0)
        ).alias("Gross_Revenue"),

        sum(
            when(col("Total") < 0, -col("Total"))
            .otherwise(0)
        ).alias("Returns"),

        sum("Total").alias("Net_Revenue"),

        countDistinct(
            when(
                col("Quantity") > 0,
                col("InvoiceNo")
            )
        ).alias("Orders"),

        countDistinct(
            when(
                col("CustomerID").isNotNull(),
                col("CustomerID")
            )
        ).alias("Customers"),

        sum(
            when(col("Quantity") > 0, col("Quantity"))
            .otherwise(0)
        ).alias("Units_Sold"),

        avg(
            when(
                col("Quantity") > 0,
                col("Total")
            )
        ).alias("Average_Line_Value")
    )
    .withColumn(
        "Average_Net_Order_Value",
        col("Net_Revenue") / col("Orders")
    )
)

gold_overall_kpis.show()

gold_overall_kpis.write.format("delta").mode("overwrite").saveAsTable("gold_overall_kpis")

+--------------------+-----------------+----------------+------+---------+----------+------------------+-----------------------+
|       Gross_Revenue|          Returns|     Net_Revenue|Orders|Customers|Units_Sold|Average_Line_Value|Average_Net_Order_Value|
+--------------------+-----------------+----------------+------+---------+----------+------------------+-----------------------+
|1.0642110804003006E7|893979.7299999989|9748131.07400267| 20726|     4372|   5645017|20.230149878724927|     470.33344948386906|
+--------------------+-----------------+----------------+------+---------+----------+------------------+-----------------------+

